[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skarma91/logicmojo-ai-july-2026/blob/main/modules/module-2-ml-dl-essentials/05-training-in-practic/micro-assignment/solution/solution.ipynb)


# Micro-assignment 2.5 solution: Training in practice

Reference solution. Compare your output to these values.

### Problem 1: A DataLoader batch

In [1]:
import torch
from torch.utils.data import TensorDataset, DataLoader
X = torch.randn(100, 8)
y = torch.randint(0, 3, (100,))
loader = DataLoader(TensorDataset(X, y), batch_size=16)
xb, yb = next(iter(loader))
print("batch X shape:", tuple(xb.shape))
print("batch y shape:", tuple(yb.shape))

batch X shape: (16, 8)
batch y shape: (16,)


### Problem 2: Train and evaluate

In [2]:
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

Xd, yd = load_digits(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(Xd, yd, test_size=0.25, random_state=0, stratify=yd)
sc = StandardScaler().fit(Xtr)
Xtr = torch.tensor(sc.transform(Xtr), dtype=torch.float32)
Xte = torch.tensor(sc.transform(Xte), dtype=torch.float32)
ytr = torch.tensor(ytr); yte = torch.tensor(yte)

torch.manual_seed(0)
net = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
opt = torch.optim.Adam(net.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()
for _ in range(150):
    opt.zero_grad(); loss_fn(net(Xtr), ytr).backward(); opt.step()

acc = (net(Xte).argmax(1) == yte).float().mean().item()
print("test accuracy above 0.90:", acc > 0.90)

test accuracy above 0.90: True


### Problem 3: Save and reload give identical predictions

In [3]:
import tempfile, os
model = nn.Linear(5, 2)
path = os.path.join(tempfile.gettempdir(), "m25.pt")
torch.save(model.state_dict(), path)
fresh = nn.Linear(5, 2)
fresh.load_state_dict(torch.load(path))
xb = torch.randn(4, 5)
print("identical:", torch.equal(model(xb), fresh(xb)))

identical: True


### Problem 4: Read the overfitting onset

In [4]:
val_losses = [0.90, 0.61, 0.48, 0.52, 0.70]
best = min(range(len(val_losses)), key=lambda i: val_losses[i])
print("best epoch:", best)

best epoch: 2


### Problem 5: A mismatch is a RuntimeError

In [5]:
a = torch.zeros(2, 3)
b = torch.zeros(2, 3)
try:
    a @ b                       # (2,3) @ (2,3) does not line up
except RuntimeError as e:
    print("error type:", type(e).__name__)
result = a @ b.T                # fixed: (2,3) @ (3,2)
print("after fix, shape:", tuple(result.shape))
print("note: a device mismatch (model on GPU, batch on CPU) raises the same RuntimeError; the fix is to move both to the same device")

error type: RuntimeError
after fix, shape: (2, 2)
note: a device mismatch (model on GPU, batch on CPU) raises the same RuntimeError; the fix is to move both to the same device


### Problem 6: Reproducibility

In [6]:
def first_val(seed):
    torch.manual_seed(seed)
    return float(torch.rand(1))
print("runs match:", first_val(0) == first_val(0))

runs match: True
